In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import json
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.environ.get('API_KEY')

client = OpenAI(api_key=api_key)

from src.utils import pairwise_cosine_similarity
from sentence_transformers import SentenceTransformer
# load embedding model
embedding_model_path = "BAAI/bge-large-en-v1.5"
embedding_model = SentenceTransformer(embedding_model_path)

In [2]:
def get_gender(abstract):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": [
                    {
                    "type": "text",
                    "text": "You are a biomedical natural language processing assistant. Given the abstract of a clinical trial study, your task is to identify the gender of the study population.\n\nYour output must be in the following JSON format:\n{\n  \"gender\": \"female\"  // or \"male\" or \"neutral\"\n}\n\nGuidelines:\n- If the abstract mentions that the study participants are women or females, output \"female\".\n- If the abstract mentions men or males, output \"male\".\n- If the abstract only mentions the number of participants without specifying gender, output \"neutral\".\n- If both male and female participants are mentioned and the study includes both, still output \"neutral\".\n- Do not infer gender based on disease or context. Only use explicit statements."
                    }
                ]
            },
            {
                "role": "user",
                "content": [
                    {
                    "type": "text",
                    "text": "Now process the following abstract:\n\n {}".format(abstract)
                    }
                ]
            }
        ],
        response_format={
            "type": "json_object"
        },
        temperature=1,
        max_completion_tokens=2048,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0,
        store=False
    )

    return response.choices[0].message.content

# Male

In [3]:
male_df = pd.read_csv("../data/pubmed_rct/cav_files/penalty=1.2/all_male_results.tsv", sep="\t")

In [4]:
# alpha = 0
abstracts = male_df["abstract"].to_list()
gender_results = []
for abstract in abstracts:
    j_output = get_gender(abstract)
    gender_results.append(json.loads(j_output).get("gender"))

male_df["gender_{}".format(0)] = gender_results

In [5]:
alpha_list = [-0.0625, -0.125, -0.25, -0.5, -1, -2, -4, -8]
pbar = tqdm(total=len(male_df)*len(alpha_list))

for alpha in alpha_list:

    abstracts = male_df["abstract_{}".format(alpha)].to_list()
    
    gender_results = []
    for abstract in abstracts:
        j_output = get_gender(abstract)
        gender_results.append(json.loads(j_output).get("gender"))
        pbar.update(1)
    
    male_df["gender_{}".format(alpha)] = gender_results

pbar.close()

100%|██████████| 400/400 [04:58<00:00,  1.34it/s]


In [15]:
male_df.to_csv("../data/pubmed_rct/cav_files/penalty=1.2/all_male_results.tsv", sep="\t", index=False)

In [6]:
male_df["gender_0"].value_counts()

gender_0
male    50
Name: count, dtype: int64

In [7]:
male_df["gender_-0.0625"].value_counts()

gender_-0.0625
male       24
neutral    18
female      8
Name: count, dtype: int64

In [8]:
male_df["gender_-0.125"].value_counts()

gender_-0.125
female     27
neutral    13
male       10
Name: count, dtype: int64

In [9]:
male_df["gender_-0.25"].value_counts()

gender_-0.25
female     46
neutral     4
Name: count, dtype: int64

In [10]:
male_df["gender_-0.5"].value_counts()

gender_-0.5
female     48
neutral     2
Name: count, dtype: int64

In [11]:
male_df["gender_-1"].value_counts()

gender_-1
female    50
Name: count, dtype: int64

In [12]:
male_df["gender_-2"].value_counts()

gender_-2
female    50
Name: count, dtype: int64

In [13]:
male_df["gender_-4"].value_counts()

gender_-4
female    50
Name: count, dtype: int64

In [14]:
male_df["gender_-8"].value_counts()

gender_-8
female    50
Name: count, dtype: int64

In [16]:
male_reference_embs = embedding_model.encode(male_df["abstract"].to_list())
male_reference_embs.shape

(50, 1024)

In [17]:
alpha_list = [-0.0625, -0.125, -0.25, -0.5, -1, -2, -4, -8]
for alpha in alpha_list:
    abstracts = male_df["abstract_{}".format(alpha)].to_list()
    test_embs = embedding_model.encode(abstracts)
    cosine_list = pairwise_cosine_similarity(male_reference_embs, test_embs)
    print(alpha, np.mean(cosine_list), np.std(cosine_list))

-0.0625 0.8666507565975189 0.04342383402842999
-0.125 0.8603982245922088 0.04116771866812727
-0.25 0.8518539500236512 0.044285447512992335
-0.5 0.8259911382198334 0.047478250887356126
-1 0.7645939338207245 0.04796228114465022
-2 0.674340214729309 0.040209633119453136
-4 0.6357142996788024 0.03478043931695615
-8 0.6313473320007325 0.039849839865622454


# Female

In [19]:
female_df = pd.read_csv("../data/pubmed_rct/cav_files/penalty=1.2/all_female_results.tsv", sep="\t")

In [20]:
# alpha = 0
abstracts = female_df["abstract"].to_list()
    
gender_results = []
for abstract in abstracts:
    j_output = get_gender(abstract)
    gender_results.append(json.loads(j_output).get("gender"))

female_df["gender_{}".format(0)] = gender_results

In [21]:
alpha_list = [0.0625, 0.125, 0.25, 0.5, 1, 2, 4, 8]
pbar = tqdm(total=len(female_df)*len(alpha_list))

for alpha in alpha_list:

    abstracts = female_df["abstract_{}".format(alpha)].to_list()
    
    gender_results = []
    for abstract in abstracts:
        j_output = get_gender(abstract)
        gender_results.append(json.loads(j_output).get("gender"))
        pbar.update(1)
    
    female_df["gender_{}".format(alpha)] = gender_results

pbar.close()

100%|██████████| 400/400 [04:12<00:00,  1.58it/s]


In [22]:
female_df.to_csv("../data/pubmed_rct/cav_files/penalty=1.2/all_female_results.tsv", sep="\t", index=False)

In [23]:
female_df["gender_0"].value_counts()

gender_0
female    50
Name: count, dtype: int64

In [24]:
female_df["gender_0.0625"].value_counts()

gender_0.0625
male       21
female     15
neutral    14
Name: count, dtype: int64

In [25]:
female_df["gender_0.125"].value_counts()

gender_0.125
male       25
neutral    21
female      4
Name: count, dtype: int64

In [26]:
female_df["gender_0.25"].value_counts()

gender_0.25
male       37
neutral    13
Name: count, dtype: int64

In [27]:
female_df["gender_0.5"].value_counts()

gender_0.5
male       44
neutral     6
Name: count, dtype: int64

In [28]:
female_df["gender_1"].value_counts()

gender_1
male       48
neutral     2
Name: count, dtype: int64

In [29]:
female_df["gender_2"].value_counts()

gender_2
male       48
neutral     2
Name: count, dtype: int64

In [30]:
female_df["gender_4"].value_counts()

gender_4
male       43
neutral     7
Name: count, dtype: int64

In [31]:
female_df["gender_8"].value_counts()

gender_8
male       39
neutral    11
Name: count, dtype: int64

In [32]:
female_reference_embs = embedding_model.encode(female_df["abstract"].to_list())
alpha_list = [0.0625, 0.125, 0.25, 0.5, 1, 2, 4, 8]
for alpha in alpha_list:
    abstracts = female_df["abstract_{}".format(alpha)].to_list()
    test_embs = embedding_model.encode(abstracts)
    cosine_list = pairwise_cosine_similarity(female_reference_embs, test_embs)
    print(alpha, np.mean(cosine_list), np.std(cosine_list))

0.0625 0.8653559863567353 0.04028439115997879
0.125 0.8686985230445862 0.04072256978894935
0.25 0.8570036864280701 0.041885198842386624
0.5 0.8366067326068878 0.04385975586578645
1 0.8014882028102874 0.05123159992419134
2 0.750124204158783 0.0456042796917457
4 0.6907142877578736 0.04707521732139805
8 0.6656762146949768 0.04310676246052803
